# 🧪 Thực Nghiệm Độc Lập: Chứng Minh 3 Điểm Yếu Của LiDAR & Tính Cần Thiết Của Smoothed Surrogate
### Khung Lý Thuyết: Dimension-Free Lipschitz Bound & Randomized Smoothing (RS-LiDAR)

Notebook này thực thi **Bộ 3 Bài Test Chuẩn Xác 100% Theo Khung Lý Thuyết** để chứng minh sự vượt trội của phương pháp **Smoothed Surrogate ($r_\sigma$)** so với **LiDAR gốc (ICML 2026 Spotlight)**:
1. **Bài Test 1 (Solver Error Robustness)**: Chứng minh sai số reward của LiDAR bị bùng nổ khi dùng DPM-Solver 5 bước, trong khi Smoothed Surrogate có chặn Lipschitz $\|\nabla r_\sigma\|_2 \le L_\sigma < \infty$.
2. **Bài Test 2 (Softmax Mode Collapse)**: Đo độ sụp đổ Entropy Shannon $H(w^r)$, chứng minh LiDAR gốc bị dồn 95% trọng số vào 1 hạt duy nhất (*Best-of-1 Trap*), trong khi phương pháp của bạn phân bổ mượt mà.
3. **Bài Test 3 (Guidance Field Stability)**: Đo độ ổn định góc quay Cosine $\text{CosSim}(\mathbf{g}_t, \mathbf{g}_{t+\delta})$ khi có nhiễu vi mô $\delta = 10^{-3}$, chứng minh vector dẫn đường của bạn kháng nhiễu tuyệt đối ($\approx 0.99$).


## 1. Kiểm Tra Thiết Bị GPU (1 GPU hoặc 2 GPU)


In [ ]:
import torch
print(f"CUDA Khả dụng: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    raise RuntimeError("❌ SESSION ĐANG CHẠY TRÊN CPU!\n👉 Vui lòng vào panel bên phải Kaggle: Session options -> Accelerator -> Chọn 'GPU T4 x2' rồi chạy lại!")
n_gpus = torch.cuda.device_count()
print(f"Số lượng GPU khả dụng: {n_gpus}")
for i in range(n_gpus):
    print(f" - GPU {i}: {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory / (1024**3):.2f} GB VRAM)")


## 2. Thiết Lập Môi Trường & Trình Điều Khiển Đa GPU Song Song


In [ ]:
import os, sys, subprocess, threading
WORKDIR = "/kaggle/working/RS-LiDAR"

# Clone hoặc cập nhật mã nguồn mới nhất
!git clone https://github.com/leekwanreal/RS-LiDAR.git {WORKDIR} 2>/dev/null || (cd {WORKDIR} && git pull)
os.chdir(WORKDIR)
if WORKDIR not in sys.path:
    sys.path.insert(0, WORKDIR)

# Cài đặt các thư viện cần thiết (đồng bộ chuẩn 100% với Table 2 replication)
!pip install -q --upgrade protobuf
!pip install -q transformers==4.38.2 diffusers==0.31.0 accelerate==1.2.1 safetensors huggingface-hub einops ftfy timm peft
!pip install -q git+https://github.com/openai/CLIP.git
!pip install -q git+https://github.com/THUDM/ImageReward.git
!pip install -q hpsv2 open_clip_torch filelock matplotlib tqdm scipy seaborn pandas tabulate

# Trình thực thi đa luồng in log song song cả 2 GPU
def run_commands_parallel(cmd0, cmd1):
    def stream_pipe(pipe, prefix):
        for line in iter(pipe.readline, ''):
            if line.strip():
                print(f"{prefix} {line.strip()}", flush=True)
        pipe.close()

    p0 = subprocess.Popen(cmd0, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    p1 = subprocess.Popen(cmd1, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

    t0 = threading.Thread(target=stream_pipe, args=(p0.stdout, "[GPU 0]"))
    t1 = threading.Thread(target=stream_pipe, args=(p1.stdout, "[GPU 1]"))
    t0.start(); t1.start()
    p0.wait(); p1.wait()
    t0.join(); t1.join()

print("✅ Môi trường thực nghiệm & Trình điều phối đa GPU đã sẵn sàng!")


## 3. Cấu Hình Siêu Tham Số Thực Nghiệm


In [ ]:
# ==================== CẤU HÌNH THỰC NGHIỆM ====================
NUM_GPUS = 2             # Đặt = 2 nếu dùng GPU T4 x2 trên Kaggle, hoặc = 1 nếu chỉ dùng 1 GPU
NUM_PROMPTS = 20         # Số lượng prompt lấy mẫu ngẫu nhiên (chạy nhanh: 10-20, đầy đủ: 50)
NUM_PARTICLES = 20       # Số lượng hạt trên mỗi prompt
SIGMA = 0.30             # Độ lệch chuẩn làm mịn mặc định (sigma = 0.30)
TUNE_SIGMA = True        # Đặt = True để khảo sát ảnh hưởng của 3 mốc sigma (Ablation Study)
SIGMAS_SWEEP = "0.15,0.30,0.60" # Đúng 3 mốc: Thấp (0.15) - Tối ưu (0.30) - Cao (0.60)
TEST_MODE = "1"          # Tập trung chạy Test 1: Kháng sai số bộ giải & Khảo sát Sigma
OUTPUT_DIR = "/kaggle/working/experiments/test_results"

print(f"Cấu hình Số GPU: {NUM_GPUS} GPU")
print(f"Cấu hình bài test: TEST 1 (SOLVER ROBUSTNESS & SIGMA ABLATION)")
print(f"Số lượng prompt: {NUM_PROMPTS} prompts")
print(f"Số lượng hạt n: {NUM_PARTICLES}")
print(f"Độ lệch chuẩn sigma mặc định: {SIGMA}")
print(f"3 Mốc Khảo sát Sigma: {SIGMAS_SWEEP}")
print(f"Thư mục lưu kết quả: {OUTPUT_DIR}")


## 4. Chạy Bộ 3 Bài Test Khoa Học (Hỗ Trợ 2 GPU Song Song)


In [ ]:
import os
os.chdir(WORKDIR)

tune_arg = f"--tune_sigma --sigmas=\"{SIGMAS_SWEEP}\"" if TUNE_SIGMA else ""

if NUM_GPUS == 2:
    print(f"🚀 [2 GPU] Đang chạy song song Bộ 3 Bài Test trên GPU 0 và GPU 1 (mỗi GPU 1/2 số prompt)...")
    cmd0 = f"""python test_lidar_weaknesses.py \
        --test={TEST_MODE} \
        --num_prompts={NUM_PROMPTS} \
        --num_particles={NUM_PARTICLES} \
        --sigma={SIGMA} {tune_arg} \
        --output_dir="{OUTPUT_DIR}" \
        --gpu_id=0 --num_shards=2 --shard_id=0"""
    cmd1 = f"""python test_lidar_weaknesses.py \
        --test={TEST_MODE} \
        --num_prompts={NUM_PROMPTS} \
        --num_particles={NUM_PARTICLES} \
        --sigma={SIGMA} {tune_arg} \
        --output_dir="{OUTPUT_DIR}" \
        --gpu_id=1 --num_shards=2 --shard_id=1"""
    run_commands_parallel(cmd0, cmd1)
    
    # Tổng hợp biểu đồ & số liệu đa GPU
    print("\n📊 Đang tổng hợp biểu đồ chung từ 2 GPU...")
    !python -c "from test_lidar_weaknesses import plot_and_save_all; plot_and_save_all(output_dir='{OUTPUT_DIR}', sigma={SIGMA})"
else:
    print(f"🚀 [1 GPU] Đang chạy Bộ 3 Bài Test trên 1 GPU...")
    !python test_lidar_weaknesses.py \
        --test={TEST_MODE} \
        --num_prompts={NUM_PROMPTS} \
        --num_particles={NUM_PARTICLES} \
        --sigma={SIGMA} {tune_arg} \
        --output_dir="{OUTPUT_DIR}"

print("\n✅ Hoàn tất toàn bộ thực nghiệm!")


## 5. Trực Quan Hóa Biểu Đồ So Sánh Khoa Học (3-Panel Publication Plot)


In [ ]:
from IPython.display import Image, display
chart_path = f"{OUTPUT_DIR}/golden_3_tests_comparison.png"
abl_chart_path = f"{OUTPUT_DIR}/sigma_ablation_curves.png"

if os.path.exists(chart_path):
    print("📊 BIỂU ĐỒ SO SÁNH 3 BÀI TEST CHUẨN XUẤT BẢN:")
    display(Image(filename=chart_path))

if os.path.exists(abl_chart_path):
    print("\n📊 ĐỒ THỊ KHẢO SÁT ẢNH HƯỞNG BÁN KÍNH LÀM MỊN SIGMA (ABLATION CURVES):")
    display(Image(filename=abl_chart_path))
elif not os.path.exists(chart_path):
    print(f"Chưa tìm thấy biểu đồ tại {chart_path}. Vui lòng chạy Cell 4 trước.")


## 6. Tổng Hợp Số Liệu Định Lượng & Xuất Bảng Kết Quả


In [ ]:
import json
import os
import pandas as pd
from IPython.display import display, Markdown

json_path = f"{OUTPUT_DIR}/summary_results.json"
csv_path = f"{OUTPUT_DIR}/weaknesses_comparison_table.csv"
md_path = f"{OUTPUT_DIR}/weaknesses_comparison_table.md"
abl_csv = f"{OUTPUT_DIR}/sigma_ablation_table.csv"
abl_md = f"{OUTPUT_DIR}/sigma_ablation_table.md"
abl_chart_path = f"{OUTPUT_DIR}/sigma_ablation_curves.png"

if os.path.exists(json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        summary = json.load(f)
    
    t1 = summary.get("test_1_solver_error", {})
    t2 = summary.get("test_2_entropy", {})
    t3 = summary.get("test_3_cosine_stability", {})
    
    # 1. BẢNG 1: THỰC NGHIỆM ĐA MÔ HÌNH REWARD (TEST 1)
    m_dict = t1.get("metrics", {})
    if not m_dict:
        m_dict = {"ImageReward": {"delta_lidar": t1.get("mean_delta_r_lidar", 0.0), "delta_ours": t1.get("mean_delta_r_ours", 0.0), "tau_lidar": t1.get("tau_lidar", 0.0), "tau_ours": t1.get("tau_ours", 0.0), "lipschitz_bound": t1.get("lipschitz_bound", 0.0)}}
    
    t1_rows = []
    for m_name, m_data in m_dict.items():
        d_l = m_data.get("delta_lidar", 0.0)
        d_o = m_data.get("delta_ours", 0.0)
        t_l = m_data.get("tau_lidar", 0.0)
        t_o = m_data.get("tau_ours", 0.0)
        l_b = m_data.get("lipschitz_bound", 0.0)
        err_red = max(0.0, (d_l - d_o) / max(1e-6, d_l) * 100) if d_l > 0 else 0.0
        tau_gain = max(0.0, (t_o - t_l) / max(1e-6, abs(t_l)) * 100) if t_l != 0 else 0.0
        t1_rows.append({
            "Mô Hình Reward": m_name,
            "Sai Số |Δr| LiDAR (σ=0) ↓": f"{d_l:.4f}",
            "Sai Số |Δr| Bạn (r_σ) ↓": f"{d_o:.4f}",
            "Giảm Sai Số (%)": f"-{err_red:.1f}%",
            "Kendall's τ LiDAR ↑": f"{t_l:.4f}",
            "Kendall's τ Bạn ↑": f"{t_o:.4f}",
            "Tăng Thứ Bậc (%)": f"+{tau_gain:.1f}%",
            "Chặn Lipschitz L_σ": f"<= {l_b:.2f}"
        })
    df_t1 = pd.DataFrame(t1_rows)
    
    # 2. BẢNG 2: ĐỘNG LỰC HỌC PHÂN PHỐI & ĐỘ ỔN ĐỊNH VECTOR (TEST 2 & 3)
    t23_rows = []
    if t2:
        e_l = t2.get("entropy_lidar_mean", 0.0)
        e_o = t2.get("entropy_ours_mean", 0.0)
        t23_rows.append({
            "Tiêu Chí Đánh Giá": "Test 2: Entropy Softmax H(w^r) (50 hạt) ↑",
            "LiDAR Gốc (σ=0)": f"{e_l:.4f} bits (N_eff={2**e_l:.1f})",
            "Phương Pháp Của Bạn (r_σ)": f"{e_o:.4f} bits (N_eff={2**e_o:.1f})",
            "Mức Cải Thiện": f"+{e_o - e_l:.4f} bits",
            "Ý Nghĩa Khoa Học": "Chống sụp đổ One-Hot (Best-of-1 Trap), huy động đa hạt"
        })
    if t3:
        c_l = t3.get("cossim_lidar_mean", 0.0)
        c_o = t3.get("cossim_ours_mean", 0.0)
        t23_rows.append({
            "Tiêu Chí Đánh Giá": "Test 3: Độ Ổn Định CosSim(g_t, g_{t+δ}) (δ=1e-3) ↑",
            "LiDAR Gốc (σ=0)": f"{c_l:.4f}",
            "Phương Pháp Của Bạn (r_σ)": f"{c_o:.4f}",
            "Mức Cải Thiện": f"+{(c_o - c_l)*100:.2f}%",
            "Ý Nghĩa Khoa Học": "Vector dẫn đường Lipschitz kháng nhiễu vi mô tuyệt đối"
        })
    df_t23 = pd.DataFrame(t23_rows)
    
    print("\n================ 📊 BẢNG 1: KHÁNG SAI SỐ BỘ GIẢI TRÊN ĐA MÔ HÌNH REWARD (TEST 1) ================\n")
    display(df_t1)
    print("\n================ 📊 BẢNG 2: ĐỘNG LỰC HỌC PHÂN PHỐI & ĐỘ ỔN ĐỊNH VECTOR (TEST 2 & 3) ================\n")
    display(df_t23)
    
    # 3. BẢNG 3: KHẢO SÁT ẢNH HƯỞNG BÁN KÍNH LÀM MỊN SIGMA (ABLATION STUDY)
    if os.path.exists(abl_csv):
        df_abl = pd.read_csv(abl_csv)
        print("\n================ 📊 BẢNG 3: KHẢO SÁT ẢNH HƯỞNG BÁN KÍNH LÀM MỊN SIGMA (ABLATION STUDY) ================\n")
        display(df_abl)
        os.system(f"cp {abl_csv} /kaggle/working/sigma_ablation_table.csv")
        os.system(f"cp {abl_md} /kaggle/working/sigma_ablation_table.md")
        if os.path.exists(abl_chart_path):
            os.system(f"cp {abl_chart_path} /kaggle/working/sigma_ablation_curves.png")
    
    # Xuất ra thư mục /kaggle/working để người dùng tải về
    if os.path.exists(csv_path):
        os.system(f"cp {csv_path} /kaggle/working/weaknesses_comparison_table.csv")
        os.system(f"cp {md_path} /kaggle/working/weaknesses_comparison_table.md")
        print("\n💾 ĐÃ LƯU TẤT CẢ BẢNG KẾT QUẢ VÀO /kaggle/working ĐỂ TẢI VỀ:")
        print(" • CSV Tổng hợp:         /kaggle/working/weaknesses_comparison_table.csv")
        print(" • Markdown Tổng hợp:    /kaggle/working/weaknesses_comparison_table.md")
        if os.path.exists(abl_csv):
            print(" • CSV Khảo sát Sigma:   /kaggle/working/sigma_ablation_table.csv")
            print(" • Markdown Khảo sát:    /kaggle/working/sigma_ablation_table.md")
            print(" • Đồ thị Khảo sát:      /kaggle/working/sigma_ablation_curves.png")
else:
    print(f"Chưa tìm thấy file tổng hợp tại {json_path}.")
